

## Configuração das bibliotecas



In [14]:
# importação das bibliotecas

import requests
import pandas as pd
import datetime as dt

## Configurações padrões para o notebook

In [31]:
TIMEOUT = 60
BASE_URL = 'https://atlas.ripe.net/api/v2'
ANCHORS_URL = 'anchors'
MEASUREMENTS_URL = 'measurements'
PROBES_URL = 'probes'

## Exibição dos Anchor IDs

In [24]:
# CONFIGURAÇÃO

# Coloque a código do país que deseja visualizar
COUNTRY = 'BR'

URL = f'{BASE_URL}/{ANCHORS_URL}'
# parametros da requisição
params = {
    'is_active': 'true'
}

if COUNTRY is not None:
    params['country'] = COUNTRY.upper()

response = requests.get(
    URL,
    params=params,
    timeout=TIMEOUT
)

print('Configuração da API')
print('URL BASE:', URL)
print('País:', COUNTRY)
print('Parâmetros:', params)
print(f'URL com parametros: {response.url}')
print('')

response.raise_for_status()
data = response.json()['results']

# Extração dos anchor ids
anchors = pd.json_normalize(data)
anchors = anchors[['id', 'probe', 'fqdn', 'ip_v4', 'country', 'city', 'company']]

print('\nResultado')
print('Resposta da API: ', response)
print('Quantidade de anchors:', len(anchors))

display(anchors)

Configuração da API
URL BASE: https://atlas.ripe.net/api/v2/anchors
País: BR
Parâmetros: {'is_active': 'true', 'country': 'BR'}
URL com parametros: https://atlas.ripe.net/api/v2/anchors?is_active=true&country=BR


Resultado
Resposta da API:  <Response [200]>
Quantidade de anchors: 15


,id,probe,fqdn,ip_v4,country,city,company
0,2334,6891,br-bhz-as10417-client.anchors.atlas.ripe.net,150.164.1.222,BR,Belo Horizonte,Universidade Federal de Minas Gerais
1,4093,7457,br-cci-as61618.anchors.atlas.ripe.net,170.81.100.114,BR,Camaçari,Viva Tecnologia Telecom
2,3679,7307,br-fec-as264479.anchors.atlas.ripe.net,138.117.32.11,BR,Feira de Santana,Turbozone Internet
3,3497,7242,br-for-as37468.anchors.atlas.ripe.net,170.238.235.214,BR,Fortaleza,Angola Cables
4,2849,7019,br-pgz-as53046.anchors.atlas.ripe.net,143.208.160.10,BR,Ponta Grossa,Universidade Estadual de Ponta Grossa
5,4331,7568,br-pil-as52564.anchors.atlas.ripe.net,177.85.156.51,BR,Primavera do Leste,Biazi Telecom
6,1164,6349,br-poa-as19200.anchors.atlas.ripe.net,200.132.1.32,BR,Porto Alegre,Federal University of Rio Grande do Sul
7,4185,7508,br-qoc-as14907.anchors.atlas.ripe.net,195.200.68.66,BR,Osasco,Wikimedia Foundation
8,2474,6977,br-sao-as12008.anchors.atlas.ripe.net,156.154.122.254,BR,São Paulo,DigiCert
9,1974,6659,br-sao-as199524.anchors.atlas.ripe.net,5.188.238.222,BR,São Paulo,G-Core Labs S.A.


# Exibição das medidas na qual o Anchor ID escolhido anteriormente é destino

In [25]:
# ip do anchor selecionado na etapa anterior
TARGET_IP = '150.164.1.222'

STATUS = 'ongoing'
MEASUREMENT_TYPE = 'ping'
ADDRESS_FAMILY = '4'
URL = f'{BASE_URL}/{MEASUREMENTS_URL}'

# parametros da requisição
params = {
    'target_ip': TARGET_IP,
    'status': STATUS,
    'type': MEASUREMENT_TYPE,
    'af': ADDRESS_FAMILY
}

response = requests.get(
    URL,
    params=params,
    timeout=TIMEOUT
)

print('Configuração da API')
print('URL BASE:', URL)
print('Target IP:', TARGET_IP)
print('Status:', STATUS)
print('Tipo:', MEASUREMENT_TYPE)
print('Parâmetros:', params)
print(f'URL com parametros: {response.url}')
print('')

response.raise_for_status()
data = response.json()['results']

# extração das measurements
measurements = pd.json_normalize(data)
measurements = measurements[['id', 'type', 'target', 'target_ip', 'af', 'is_public', 'status.name']]

print('\nResultado')
print('Resposta da API: ', response)
print('Quantidade de measurements:', len(measurements))

display(measurements)

Configuração da API
URL BASE: https://atlas.ripe.net/api/v2/measurements
Target IP: 150.164.1.222
Status: ongoing
Tipo: ping
Parâmetros: {'target_ip': '150.164.1.222', 'status': 'ongoing', 'type': 'ping', 'af': '4'}
URL com parametros: https://atlas.ripe.net/api/v2/measurements?target_ip=150.164.1.222&status=ongoing&type=ping&af=4


Resultado
Resposta da API:  <Response [200]>
Quantidade de measurements: 2


,id,type,target,target_ip,af,is_public,status.name
0,28095697,ping,br-bhz-as10417-client.anchors.atlas.ripe.net,150.164.1.222,4,True,Ongoing
1,28097617,ping,br-bhz-as10417-client.anchors.atlas.ripe.net,150.164.1.222,4,True,Ongoing


# Escolha das medidas e investigação dos probes IDs

In [33]:
# measurement ID selecionado na etapa anterior
MEASUREMENT_ID = 28095697

# intervalo de tempo da consulta, em minutos
MEASUREMENT_MINUTES_TIME = 200

URL = f'{BASE_URL}/{MEASUREMENTS_URL}/{MEASUREMENT_ID}/results/'


# controle da janela de tempo
stop_datetime = dt.datetime.now(dt.timezone.utc)
start_datetime = (
    stop_datetime
    - dt.timedelta(minutes=MEASUREMENT_MINUTES_TIME)
)

START_TIME = start_datetime.isoformat()
STOP_TIME = stop_datetime.isoformat()


# parametros da requisição
params = {
    'start': START_TIME,
    'stop': STOP_TIME,
    'probe_ids': '7113'
}

response = requests.get(
    URL,
    params=params,
    timeout=TIMEOUT
)

print('Configuração da API')
print('URL BASE:', URL)
print('Measurement ID:', MEASUREMENT_ID)
print('Intervalo:', f'{MEASUREMENT_MINUTES_TIME} minutos')
print('Início (UTC):', START_TIME)
print('Fim (UTC):', STOP_TIME)
print('Parâmetros:', params)
print(f'URL com parametros: {response.url}')
print('')


# extração dos dados
response.raise_for_status()
data = response.json()

print('\nResultado')
print('Resposta da API: ', response)
print('Quantidade de resultados:', len(data))

display(pd.json_normalize(data))

Configuração da API
URL BASE: https://atlas.ripe.net/api/v2/measurements/28095697/results/
Measurement ID: 28095697
Intervalo: 200 minutos
Início (UTC): 2026-09-29T20:47:11.454130+00:00
Fim (UTC): 2026-09-30T00:07:11.454130+00:00
Parâmetros: {'start': '2026-09-29T20:47:11.454130+00:00', 'stop': '2026-09-30T00:07:11.454130+00:00', 'probe_ids': '7113'}
URL com parametros: https://atlas.ripe.net/api/v2/measurements/28095697/results/?start=2026-09-29T20%3A47%3A11.454130%2B00%3A00&stop=2026-09-30T00%3A07%3A11.454130%2B00%3A00&probe_ids=7113


Resultado
Resposta da API:  <Response [200]>
Quantidade de resultados: 50


,fw,mver,lts,dst_name,af,dst_addr,src_addr,proto,ttl,size,...,avg,msm_id,prb_id,timestamp,msm_name,from,type,group_id,step,stored_timestamp
0,5130,2.6.4,253,150.164.1.222,4,150.164.1.222,177.124.129.226,ICMP,49,64,...,109.146837,28095697,7113,1790714874,Ping,177.124.129.226,ping,28095696,240,1790714912
1,5130,2.6.4,492,150.164.1.222,4,150.164.1.222,177.124.129.226,ICMP,49,64,...,112.766067,28095697,7113,1790715113,Ping,177.124.129.226,ping,28095696,240,1790715173
2,5130,2.6.4,741,150.164.1.222,4,150.164.1.222,177.124.129.226,ICMP,49,64,...,109.313372,28095697,7113,1790715362,Ping,177.124.129.226,ping,28095696,240,1790715453
3,5130,2.6.4,75,150.164.1.222,4,150.164.1.222,177.124.129.226,ICMP,49,64,...,109.203809,28095697,7113,1790715593,Ping,177.124.129.226,ping,28095696,240,1790715629
4,5130,2.6.4,321,150.164.1.222,4,150.164.1.222,177.124.129.226,ICMP,49,64,...,109.569747,28095697,7113,1790715839,Ping,177.124.129.226,ping,28095696,240,1790715878
5,5130,2.6.4,563,150.164.1.222,4,150.164.1.222,177.124.129.226,ICMP,49,64,...,109.260302,28095697,7113,1790716081,Ping,177.124.129.226,ping,28095696,240,1790716113
6,5130,2.6.4,798,150.164.1.222,4,150.164.1.222,177.124.129.226,ICMP,49,64,...,109.280974,28095697,7113,1790716316,Ping,177.124.129.226,ping,28095696,240,1790716365
7,5130,2.6.4,138,150.164.1.222,4,150.164.1.222,177.124.129.226,ICMP,49,64,...,109.264707,28095697,7113,1790716560,Ping,177.124.129.226,ping,28095696,240,1790716662
8,5130,2.6.4,377,150.164.1.222,4,150.164.1.222,177.124.129.226,ICMP,49,64,...,109.236270,28095697,7113,1790716799,Ping,177.124.129.226,ping,28095696,240,1790716901
9,5130,2.6.4,623,150.164.1.222,4,150.164.1.222,177.124.129.226,ICMP,49,64,...,109.174052,28095697,7113,1790717045,Ping,177.124.129.226,ping,28095696,240,1790717138


## Consulta de informações de um único probe individual

In [37]:
# Probe ID selecionado na etapa anterior
PROBE_ID = 7113

PROBE_API_URL = f'{BASE_URL}/{PROBES_URL}/{PROBE_ID}/'
TIMEOUT = 60

response = requests.get(
    PROBE_API_URL,
    timeout=TIMEOUT
)

print('Configuração da API')
print('URL BASE:', PROBE_API_URL)
print('Probe ID:', PROBE_ID)
print('')

response.raise_for_status()
data = response.json()


# extração dos dados
probe = pd.json_normalize(data)

probe = probe[
    [
        'id',
        'country_code',
        'address_v4',
        'is_public'
    ]
]

print('\nResultado')
print('Resposta da API: ', response)

display(probe)

Configuração da API
URL BASE: https://atlas.ripe.net/api/v2/probes/7113/
Probe ID: 7113


Resultado
Resposta da API:  <Response [200]>


,id,country_code,address_v4,is_public
0,7113,BR,177.124.129.226,True
